# Notebook 05 — Attention and Transformers

**Module:** 14 — Deep Learning and GNNs  
**Tier:** 2 — Working competence  
**Notebook:** 5 of 12  
**Time estimate:** 90 minutes

> Attention is a soft, differentiable dictionary lookup.
> The transformer is a stack of attention layers. ESM-2 (the protein language model)
> is a transformer trained on UniProt. AlphaFold's evoformer is a transformer
> that does attention over both sequences and residue pairs simultaneously.

---
## Step 1 — Motivation

A bidirectional LSTM at position 50 still has to propagate information through
50 sequential steps from position 1. Attention lets every position attend *directly*
to every other position in one step — constant-distance regardless of sequence length.
This is why transformers scale to 1000-residue proteins and 196kb genomic windows
where LSTMs fail.

---
## Step 2 — Intuition

**Attention as a soft lookup:**
Imagine a database of (key, value) pairs. To answer a query:
1. Compute similarity of query to each key
2. Softmax the similarities to get a probability distribution (attention weights)
3. Return the weighted sum of values

In a transformer: queries, keys, and values are all linear projections of the
same input sequence. Each position computes a query, asks "which other positions
are relevant to me?", and assembles information from them.

**Multi-head attention:**
Run $h$ attention heads in parallel, each with its own Q/K/V projections.
Different heads can learn different types of relationships:
one head might track local structure, another might capture global context.

**Positional encoding:**
Attention is permutation-invariant (doesn't know position order).
Add sinusoidal or learned positional embeddings to the input to inject position.

**Transformer encoder (for sequence classification):**
Stack of (MultiHeadAttention → Add&Norm → FFN → Add&Norm).
Read the CLS token output for classification.
Or pool across all positions.

---
## Step 3 — Biological Background

**ESM-2 (Evolutionary Scale Modeling, Lin 2023, Science):**
- 33-layer transformer trained on 250M protein sequences (UniRef90)
- Learns to predict masked residues (masked language model, like BERT)
- Learns coevolutionary patterns that correlate with 3D contact maps
- ESM-2 embeddings used directly for structure prediction, property prediction,
  protein design without fine-tuning

**DNABERT (Ji 2021, Bioinformatics):**
- BERT applied to DNA sequences (6-mer tokenization)
- Pre-trained on human reference genome
- Fine-tuned for promoter prediction, splice site detection, TF binding

**Enformer (Avsec 2021):**
- Predicts 5,313 genomic tracks from 196kb sequence windows
- CNN first layers extract local features → Transformer integrates long-range context
- Attention heads capture distal regulatory elements (enhancer → promoter, 100kb apart)

**AlphaFold evoformer:**
- Operates on two representations: sequence pair (MSA) and residue pair (distances)
- Row attention: each residue attends to other residues
- Column attention: each alignment position attends across the MSA
- "Triangle" updates: propagate distance information consistently around triplets

---
## Step 4 — Mathematical Explanation

**Scaled dot-product attention:**
$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- $Q \in \mathbb{R}^{n \times d_k}$: queries (one per sequence position)
- $K \in \mathbb{R}^{n \times d_k}$: keys
- $V \in \mathbb{R}^{n \times d_v}$: values
- Scaling by $\sqrt{d_k}$: prevents dot products from growing too large in magnitude,
  which would push softmax into saturation (zero-gradient) regions

**Multi-head attention:**
$$\text{MHA}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(XW_i^Q, XW_i^K, XW_i^V)$$

**Transformer encoder block:**
$$X' = \text{LayerNorm}(X + \text{MHA}(X))$$
$$X'' = \text{LayerNorm}(X' + \text{FFN}(X'))$$
$$\text{FFN}(x) = \text{max}(0, xW_1 + b_1)W_2 + b_2$$

**Sinusoidal positional encoding:**
$$PE(pos, 2i) = \sin(pos / 10000^{2i/d_{model}})$$
$$PE(pos, 2i+1) = \cos(pos / 10000^{2i/d_{model}})$$

In [ ]:
# Step 6 — Python: Attention from scratch + small transformer encoder
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42); np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Scaled dot-product attention from scratch ----
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (batch, n_heads, seq, d_k)
    K: (batch, n_heads, seq, d_k)
    V: (batch, n_heads, seq, d_v)
    Returns: (batch, n_heads, seq, d_v), (batch, n_heads, seq, seq)
    """
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (b, h, seq, seq)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)  # (b, h, seq, seq)
    output = torch.matmul(attn_weights, V)    # (b, h, seq, d_v)
    return output, attn_weights

# Quick verification
B, H, N, dk = 2, 4, 10, 16
Q = torch.randn(B, H, N, dk)
K = torch.randn(B, H, N, dk)
V = torch.randn(B, H, N, dk)
out, attn = scaled_dot_product_attention(Q, K, V)
print(f'Attention output shape: {out.shape}  (expected {(B,H,N,dk)})')
print(f'Attention weights sum: {attn.sum(-1).mean():.4f}  (expected 1.0 per row)')

# ---- Transformer encoder for sequence classification ----
class TransformerEncoder(nn.Module):
    def __init__(self, input_dim, d_model=64, n_heads=4, n_layers=2, ffn_dim=128,
                   dropout=0.1, max_len=100, n_classes=2):
        super().__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        # Sinusoidal positional encoding
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=ffn_dim,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier = nn.Linear(d_model, n_classes)
    
    def forward(self, x):
        # x: (batch, seq, input_dim)
        seq_len = x.shape[1]
        h = self.embedding(x) + self.pe[:, :seq_len, :]  # add positional encoding
        h = self.transformer(h)  # (batch, seq, d_model)
        h_pooled = h.mean(dim=1)  # mean pooling across sequence
        return self.classifier(h_pooled)

# ---- Dataset: TF binding (from NB03) ----
NUCL = 'ACGT'
NUCL_IDX = {b: i for i, b in enumerate(NUCL)}
MOTIF = 'CCGCGNGGNGG'
MOTIF_L = 11
SEQ_L = 50

rng_data = np.random.default_rng(42)
def generate_seqs(n=1200):
    seqs, labels = [], []
    for i in range(n):
        seq = list(''.join(rng_data.choice(list(NUCL), SEQ_L)))
        if i < n//2:  # positive
            pos = rng_data.integers(0, SEQ_L - MOTIF_L)
            for j, b in enumerate(MOTIF):
                if b != 'N':
                    seq[pos+j] = b if rng_data.random() > 0.1 else rng_data.choice(list(NUCL))
            labels.append(1)
        else:
            labels.append(0)
        seqs.append(''.join(seq))
    return seqs, np.array(labels)

seqs, y_seq = generate_seqs(1200)

def one_hot_seq(seq):
    X = np.zeros((len(seq), 4), dtype=np.float32)  # (L, 4) for transformer
    for i, b in enumerate(seq):
        if b in NUCL_IDX:
            X[i, NUCL_IDX[b]] = 1.0
    return X

class SeqDS(Dataset):
    def __init__(self, seqs, labels):
        self.X = [torch.tensor(one_hot_seq(s)) for s in seqs]
        self.y = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

ds = SeqDS(seqs, y_seq)
train_ds, val_ds = torch.utils.data.random_split(ds, [1000, 200])
train_ld = DataLoader(train_ds, batch_size=64, shuffle=True)
val_ld = DataLoader(val_ds, batch_size=128)

# ---- Train transformer ----
model_tf = TransformerEncoder(input_dim=4, d_model=64, n_heads=4, n_layers=2,
                                 ffn_dim=128, max_len=SEQ_L).to(device)
print(f'Transformer parameters: {sum(p.numel() for p in model_tf.parameters()):,}')

opt_tf = optim.Adam(model_tf.parameters(), lr=1e-3)
criterion_tf = nn.CrossEntropyLoss()

tf_losses, tf_accs = [], []
for epoch in range(40):
    model_tf.train()
    bl = []
    for X_b, y_b in train_ld:
        opt_tf.zero_grad()
        pred = model_tf(X_b.to(device))
        loss = criterion_tf(pred, y_b.to(device))
        loss.backward(); opt_tf.step()
        bl.append(loss.item())
    tf_losses.append(np.mean(bl))
    
    model_tf.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_b, y_b in val_ld:
            pred = model_tf(X_b.to(device)).argmax(-1)
            correct += (pred == y_b.to(device)).sum().item()
            total += len(y_b)
    tf_accs.append(correct/total)
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}: loss={tf_losses[-1]:.4f}, val acc={tf_accs[-1]:.3f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Training curves
ax = axes[0]
ax.plot(tf_losses, 'steelblue', lw=1.5, label='Train loss')
ax2 = ax.twinx()
ax2.plot(tf_accs, 'tomato', lw=1.5, label='Val accuracy')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss', color='steelblue')
ax2.set_ylabel('Accuracy', color='tomato')
ax.set_title('A. Transformer training\n(TF binding classification)')
lines = [plt.Line2D([0],[0],color='steelblue',lw=2,label='Train loss'),
           plt.Line2D([0],[0],color='tomato',lw=2,label='Val acc')]
ax.legend(handles=lines, fontsize=8, loc='center right')

# Panel B: Attention weight visualization for one sequence
ax = axes[1]
# Extract attention weights from the first transformer layer
pos_seq_str = seqs[0]  # first positive sequence
X_one = torch.tensor(one_hot_seq(pos_seq_str), dtype=torch.float32).unsqueeze(0).to(device)

model_tf.eval()
with torch.no_grad():
    # Hook to capture attention weights
    attn_weights_captured = []
    
    # Manual forward pass to get attention weights
    h = model_tf.embedding(X_one) + model_tf.pe[:, :SEQ_L, :]
    # Get attention from first layer
    layer0 = model_tf.transformer.layers[0]
    # Use PyTorch's built-in attention with need_weights=True
    h_norm = layer0.norm1(h)
    attn_out, attn_w = layer0.self_attn(h_norm, h_norm, h_norm, need_weights=True)
    attn_w_np = attn_w.squeeze().cpu().numpy()  # (seq, seq)

im = ax.imshow(attn_w_np, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.7)
ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
ax.set_title('B. Attention weights (layer 0, head avg)\n(brighter = higher attention)')

# Panel C: Positional encoding visualization
ax = axes[2]
pe = model_tf.pe.squeeze(0).cpu().numpy()  # (max_len, d_model)
im = ax.imshow(pe[:40, :32].T, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.7)
ax.set_xlabel('Sequence position')
ax.set_ylabel('Encoding dimension')
ax.set_title('C. Sinusoidal positional encoding\n(each position has a unique fingerprint)')

plt.suptitle('Module 14 NB05: Attention and Transformers', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('attention_transformers.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement scaled dot-product attention from scratch and verify:
   (a) output is a weighted sum of values;
   (b) attention weights sum to 1 along the key dimension.
2. What happens to attention weights if you remove the $\sqrt{d_k}$ scaling?
   Show empirically by comparing the softmax distribution with and without it
   for $d_k = 64$ with random Q, K matrices.
3. Replace mean pooling with CLS token pooling (prepend a learnable CLS embedding,
   use only the CLS position output for classification). Compare performance.
4. Visualize attention weights for a positive sequence (containing the motif).
   Do any attention heads focus on the motif region?

---
## Step 10 — Quiz

1. Write the scaled dot-product attention formula. Why is scaling by $\sqrt{d_k}$ necessary?
2. What are queries, keys, and values in the context of a transformer encoder
   applied to a protein sequence?
3. Why is positional encoding necessary? What does attention alone lack?
4. What is layer normalization and why is it used instead of batch normalization
   in transformer architectures?
5. Explain the ESM-2 pre-training objective. How does it force the model to
   learn evolutionary information?

---
## Step 12 — Reflection

> *[In one paragraph: explain why attention is O(n²) in sequence length and what
> this means for applying transformers to 196kb genomic windows like Enformer.
> How does Enformer address this computational challenge?]*

---
*Next: `06_autoencoders_and_vaes.ipynb`*